# 02. Self-Speculative Draft–Verify–Commit

**학습 목표**: diffusion drafter가 틀려도 AR verifier와 동일한 greedy token sequence만 출력되는지 확인합니다. 실제 model logit 대신 미리 정한 token으로 알고리즘의 불변조건을 검증합니다.

**실행 방법**: Python 3과 Jupyter에서 셀을 위에서 아래로 실행합니다. 외부 패키지는 필요하지 않습니다.

In [ ]:
def longest_matching_prefix(draft, verifier):
    count = 0
    for proposed, expected in zip(draft, verifier):
        if proposed != expected:
            break
        count += 1
    return count

def make_draft(target, start, block_size):
    draft = target[start:start + block_size].copy()
    # 학습용으로 특정 round에 오류를 주입합니다.
    if start == 0 and len(draft) > 2:
        draft[2] = '<잘못된-token>'
    elif start == 7 and len(draft) > 3:
        draft[3] = '<중복-token>'
    return draft


In [ ]:
target = '표 안의 숫자는 이미지 증거를 따라 정확히 전사한다'.split()
block_size = 4
output = []
rounds = []

while len(output) < len(target):
    start = len(output)
    verifier = target[start:start + block_size]
    draft = make_draft(target, start, block_size)
    accepted = longest_matching_prefix(draft, verifier)
    output.extend(verifier[:accepted])
    # 첫 불일치가 있으면 AR token 하나를 넣어 항상 전진합니다.
    if accepted < len(verifier):
        output.append(verifier[accepted])
    rounds.append({
        'start': start, 'draft': draft, 'verified': verifier,
        'draft_accepted': accepted, 'total_committed': len(output) - start,
    })

for index, row in enumerate(rounds, 1):
    print(f'round {index}:', row)
print('출력:', ' '.join(output))


## 동일성과 비용 확인

각 round는 draft와 verify 두 forward를 사용합니다. Draft가 틀린 위치는 출력되지 않으며 verifier token으로 교체됩니다.

In [ ]:
assert output == target
assert all('<' not in token for token in output)
forwards = 2 * len(rounds)
tokens_per_forward = len(output) / forwards
print('AR byte sequence와 동일:', ' '.join(output).encode() == ' '.join(target).encode())
print('round:', len(rounds), 'forward:', forwards)
print('tokens/forward:', round(tokens_per_forward, 2))


## 확장 과제

Draft 오류를 뒤쪽으로 옮겨 acceptance length를 늘리거나 block size를 바꾸세요. Sampling에서는 token 동일성 비교만으로 분포가 보존되지 않는 이유도 speculative rejection rule과 연결해 설명해 보세요.